<a href="https://colab.research.google.com/github/ever-oli/TensorPoly/blob/main/transformer_R.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Install R6 if you haven't: install.packages("R6")
library(R6)

SimpleTokenizer <- R6Class("SimpleTokenizer",
  public = list(
    word_to_id = list(),
    id_to_word = character(),
    vocab_size = 0,

    # Special tokens
    pad_token = "<PAD>",
    unk_token = "<UNK>",
    bos_token = "<BOS>",
    eos_token = "<EOS>",

    initialize = function() {
      self$word_to_id <- list()
      self$id_to_word <- character()
      self$vocab_size <- 0
    },

    #' Build vocabulary from a vector of strings
    build_vocab = function(texts) {
      # Step 1: Initialize with special tokens (R is 1-indexed, but we can use 0-indexing for IDs)
      special_tokens <- c(self$pad_token, self$unk_token, self$bos_token, self$eos_token)

      # Use a named list for word_to_id (O(1) lookup)
      for (i in seq_along(special_tokens)) {
        token <- special_tokens[i]
        self$word_to_id[[token]] <- i - 1  # 0-based IDs like Python
        self$id_to_word[i] <- token       # R vector index matches ID+1
      }

      # Step 2: Extract unique words
      # Split by whitespace, unlist to a vector, and get unique values
      all_words <- unlist(strsplit(texts, "\\s+"))
      unique_words <- sort(unique(all_words))

      # Step 3: Add unique words to vocab
      current_id <- length(special_tokens)
      for (word in unique_words) {
        if (is.null(self$word_to_id[[word]])) {
          self$word_to_id[[word]] <- current_id
          self$id_to_word[current_id + 1] <- word
          current_id <- current_id + 1
        }
      }

      self$vocab_size <- length(self$word_to_id)
    },

    #' Convert text to vector of IDs
    encode = function(text) {
      words <- unlist(strsplit(text, "\\s+"))
      unk_id <- self$word_to_id[[self$unk_token]]

      # Map words to IDs; if NULL (not found), return unk_id
      token_ids <- sapply(words, function(w) {
        id <- self$word_to_id[[w]]
        if (is.null(id)) return(unk_id) else return(id)
      })

      return(unname(token_ids))
    },

    #' Convert vector of IDs back to text
    decode = function(ids) {
      # We use ids + 1 because R vectors are 1-indexed
      words <- sapply(ids, function(id) {
        word <- self$id_to_word[id + 1]
        if (is.na(word)) return(self$unk_token) else return(word)
      })

      return(paste(words, collapse = " "))
    }
  )
)

# --- Usage Example ---
texts <- c("hello world", "hello R programming")
tokenizer <- SimpleTokenizer$new()
tokenizer$build_vocab(texts)

encoded <- tokenizer$encode("hello world from R")
print(encoded)
# [1] 4 7 1 6  (IDs may vary based on sorting, 1 is <UNK>)

decoded <- tokenizer$decode(encoded)
print(decoded)
# [1] "hello world <UNK> R"

[1] 4 7 1 6
[1] "hello world <UNK> R"


In [13]:
library(torch)

create_embedding_layer <- function(vocab_size, d_model) {
  #' Create an embedding layer using torch.
  #'
  #' Args:
  #'   vocab_size (integer): The size of the vocabulary.
  #'   d_model (integer): The dimension of the embeddings.
  #'
  #' Returns:
  #'   nn_embedding: A torch embedding layer.

  embedding_layer <- nn_embedding(vocab_size, d_model)

  # Initialize weights with a normal distribution
  # std = 1/sqrt(d_model) helps keep the scale of embeddings consistent
  nn_init_normal_(embedding_layer$weight, mean = 0.0, std = 1.0 / sqrt(d_model))

  return(embedding_layer)
}

embed_tokens <- function(embedding_layer, tokens, d_model) {
  #' Convert token indices to scaled embeddings using a torch embedding layer.
  #'
  #' Args:
  #'   embedding_layer (nn_embedding): The torch embedding layer.
  #'   tokens (torch_tensor): A 0-indexed torch tensor of token IDs.
  #'   d_model (integer): The dimension of the embeddings.
  #'
  #' Returns:
  #'   torch_tensor: A tensor of scaled embeddings for the given tokens.

  # Step 1: Look up embeddings for each token
  # The R nn_embedding expects 1-based indexing, so add 1 to 0-indexed tokens
  # Use tokens$add(1L) to ensure the operation maintains torch_long type
  embedded <- embedding_layer(tokens$add(1L)) # Shape: [..., d_model]

  # Step 2: Scale by sqrt(d_model) as per Transformer paper
  scaled_embeddings <- embedded * sqrt(d_model)

  return(scaled_embeddings)
}

# --- Usage Example ---
# Assume we have a vocab size of 100 and embedding dimension of 512
vocab_size_ex <- 100
d_model_ex <- 512

# Create the embedding layer
embedding_layer_torch <- create_embedding_layer(vocab_size_ex, d_model_ex)
print(paste("Embedding layer weight dimensions:", paste(embedding_layer_torch$weight$shape, collapse = "x")))

# Example tokens (0-indexed, as a torch tensor)
example_tokens_torch <- torch_tensor(c(0, 5, 10, 99), dtype = torch_long())

# Embed and scale tokens
embedded_result_torch <- embed_tokens(embedding_layer_torch, example_tokens_torch, d_model_ex)

print("\nFirst 5 dimensions of the first embedded token (torch):")
print(embedded_result_torch[1, 1:5])
print(paste("Embedded result dimensions (torch):", paste(embedded_result_torch$shape, collapse = "x")))


[1] "Embedding layer weight dimensions: 100x512"
[1] "\nFirst 5 dimensions of the first embedded token (torch):"
torch_tensor
 0.0712
 0.3824
-1.2647
 1.2894
 0.2177
[ CPUFloatType{5} ][ grad_fn = <SliceBackward0> ]
[1] "Embedded result dimensions (torch): 4x512"


In [14]:
library(torch)

positional_encoding <- function(seq_length, d_model) {
  #' Generate sinusoidal positional encodings using torch.
  #'
  #' Args:
  #'   seq_length (integer): The maximum sequence length.
  #'   d_model (integer): The dimension of the model (embedding dimension).
  #'
  #' Returns:
  #'   torch_tensor: A tensor of shape (seq_length, d_model) containing the positional encodings.

  # Step 1: Create position indices [0, 1, 2, ..., seq_length-1]
  position <- torch_arange(0, seq_length - 1, dtype = torch_float())$unsqueeze(2) # Shape: (seq_length, 1)

  # Step 2: Create dimension indices [0, 2, 4, ...] for the division term
  i <- torch_arange(0, d_model - 1, step = 2, dtype = torch_float()) # Shape: (d_model/2,)

  # Step 3: Calculate the division term: 10000^(2i/d_model)
  div_term <- torch_exp(i * (-log(10000.0) / d_model)) # Shape: (d_model/2,)

  # Calculate sin and cos parts
  sin_part <- torch_sin(position * div_term) # Shape: (seq_length, d_model/2)
  cos_part <- torch_cos(position * div_term) # Shape: (seq_length, d_model/2)

  # Step 4: Combine sin and cos parts to interleave them
  # Stack along a new dimension (dim=3), then view to flatten the last two dimensions
  pe_stacked <- torch_stack(list(sin_part, cos_part), dim = 3) # Shape: (seq_length, d_model/2, 2)
  pe <- pe_stacked$view(c(seq_length, d_model)) # Shape: (seq_length, d_model)

  return(pe)
}

# --- Usage Example ---
# Define parameters
seq_len_example <- 50
d_model_example <- 512

# Generate positional encodings
pe_matrix_torch <- positional_encoding(seq_len_example, d_model_example)

# Print dimensions to verify
print(paste("Positional Encoding tensor dimensions:", paste(pe_matrix_torch$shape, collapse = "x")))

# Print a slice of the tensor for inspection (e.g., first 5 rows, first 10 columns)
print("\nFirst 5 rows and first 10 columns of PE tensor:")
print(pe_matrix_torch[1:5, 1:10])


[1] "Positional Encoding tensor dimensions: 50x512"
[1] "\nFirst 5 rows and first 10 columns of PE tensor:"
torch_tensor
 0.0000  1.0000  0.0000  1.0000  0.0000  1.0000  0.0000  1.0000  0.0000  1.0000
 0.8415  0.5403  0.8219  0.5697  0.8020  0.5974  0.7819  0.6234  0.7617  0.6479
 0.9093 -0.4161  0.9364 -0.3509  0.9581 -0.2863  0.9749 -0.2227  0.9870 -0.1604
 0.1411 -0.9900  0.2451 -0.9695  0.3428 -0.9394  0.4336 -0.9011  0.5173 -0.8558
-0.7568 -0.6536 -0.6572 -0.7537 -0.5486 -0.8361 -0.4342 -0.9008 -0.3167 -0.9485
[ CPUFloatType{5,10} ]


In [4]:
#' Helper for 3D Batched Matrix Multiplication (batch, m, n) %*% (batch, n, p) -> (batch, m, p)
bmm_3d <- function(A, B) {
  # A: (batch, m, n)
  # B: (batch, n, p)
  # Output: (batch, m, p)

  dims_A <- dim(A)
  dims_B <- dim(B)

  batch_size <- dims_A[1]
  m <- dims_A[2]
  n_A <- dims_A[3]
  n_B <- dims_B[2]
  p <- dims_B[3]

  if (batch_size != dims_B[1]) {
    stop("Batch sizes must match for bmm_3d")
  }
  if (n_A != n_B) {
    stop("Inner dimensions must match for bmm_3d: n_A != n_B")
  }

  result <- array(0, dim = c(batch_size, m, p))

  for (i in 1:batch_size) {
    result[i, , ] <- A[i, , ] %*% B[i, , ]
  }
  return(result)
}

#' Softmax Activation (along the last dimension)
softmax <- function(x) {
  # x is a multi-dimensional array
  ndim <- length(dim(x))

  if (ndim == 1) {
    # For a 1D vector, apply standard softmax
    exp_x <- exp(x - max(x)) # for numerical stability
    return(exp_x / sum(exp_x))
  } else {
    # For multi-dimensional array, apply softmax along the last dimension
    # Get indices for all dimensions EXCEPT the last one
    margins_for_apply <- 1:(ndim - 1)

    # Calculate max across the last dimension for stability
    x_max <- apply(x, margins_for_apply, max)

    # Subtract x_max from x using sweep
    x_shifted <- sweep(x, margins_for_apply, x_max, FUN = "-")

    # Exponentiate
    e_x <- exp(x_shifted)

    # Sum along the last dimension
    sum_e_x <- apply(e_x, margins_for_apply, sum)

    # Divide
    result <- sweep(e_x, margins_for_apply, sum_e_x, FUN = "/")

    return(result)
  }
}

scaled_dot_product_attention <- function(Q, K, V) {
  #' Compute scaled dot-product attention.
  #'
  #' Args:
  #'   Q (array): Query tensor of shape (batch_size, seq_len_q, d_k)
  #'   K (array): Key tensor of shape (batch_size, seq_len_k, d_k)
  #'   V (array): Value tensor of shape (batch_size, seq_len_v, d_v)
  #'
  #' Returns:
  #'   array: Output tensor of shape (batch_size, seq_len_q, d_v)

  # Step 1: Get dimensions
  # Assuming Q, K, V are 3D arrays (batch, seq_len, d_model_or_k)
  d_k <- dim(Q)[length(dim(Q))] # Key/query dimension

  # Step 2: Compute attention scores
  # K.transpose(-2, -1) in R: aperm(K, c(1, 3, 2)) if K is (batch, seq_len_k, d_k)
  K_transposed <- aperm(K, c(1, 3, 2)) # Shape: (batch, d_k, seq_len_k)

  # Q: (batch, seq_len_q, d_k)
  # K_transposed: (batch, d_k, seq_len_k)
  # This results in (batch, seq_len_q, seq_len_k)
  scores <- bmm_3d(Q, K_transposed)

  # Step 3: Scale scores by 1/√d_k
  scaled_scores <- scores / sqrt(d_k)

  # Step 4: Apply softmax to get attention weights
  # Softmax is applied along the last dimension (keys dimension, seq_len_k)
  attention_weights <- softmax(scaled_scores)

  # Step 5: Compute weighted sum of values
  # attention_weights: (batch, seq_len_q, seq_len_k)
  # V: (batch, seq_len_v, d_v)
  # Assuming seq_len_k == seq_len_v for attention mechanism
  # If attention_weights is (batch, seq_len_q, seq_len_k) and V is (batch, seq_len_k, d_v)
  # then result is (batch, seq_len_q, d_v)
  output <- bmm_3d(attention_weights, V)

  return(output)
}

# --- Usage Example ---
# Assuming batch_size = 2, seq_len_q = 3, seq_len_k = 4, d_k = 5, d_v = 6
# All values are randomized for demonstration.
batch_size_ex <- 2
seq_len_q_ex <- 3
seq_len_k_ex <- 4 # Should be equal to seq_len_v_ex for standard attention
d_k_ex <- 5
d_v_ex <- 6

Q_ex <- array(rnorm(batch_size_ex * seq_len_q_ex * d_k_ex), dim = c(batch_size_ex, seq_len_q_ex, d_k_ex))
K_ex <- array(rnorm(batch_size_ex * seq_len_k_ex * d_k_ex), dim = c(batch_size_ex, seq_len_k_ex, d_k_ex))
V_ex <- array(rnorm(batch_size_ex * seq_len_k_ex * d_v_ex), dim = c(batch_size_ex, seq_len_k_ex, d_v_ex))

attention_output <- scaled_dot_product_attention(Q_ex, K_ex, V_ex)

print(paste("Output dimensions:", paste(dim(attention_output), collapse = "x")))
print("First batch, first query vector's attention output (first 5 dimensions):")
print(attention_output[1, 1, 1:min(5, d_v_ex)])


[1] "Output dimensions: 2x3x6"
[1] "First batch, first query vector's attention output (first 5 dimensions):"
[1] -0.35941089  0.06022077  0.61805267  1.44626824 -0.25218856


In [5]:
# =====================================================================
# STRICT TENSOR HELPERS
# Base R lacks native >2D matrix multiplication and row-major reshaping.
# These functions guarantee mathematical equivalence to NumPy.
# =====================================================================

#' 3D Tensor by 2D Matrix Multiplication (X @ W)
matmul_3d_2d <- function(A, B) {
  # A: (batch, seq, d_in), B: (d_in, d_out) -> returns (batch, seq, d_out)
  dims_A <- dim(A)
  if (length(dims_A) != 3) {
    stop("Input A must be a 3D array for matmul_3d_2d.")
  }

  batch_size <- dims_A[1]
  seq_len <- dims_A[2]
  d_in <- dims_A[3]
  d_out <- ncol(B)

  if (d_in != nrow(B)) {
    stop("Inner dimensions must match for matmul_3d_2d: dim(A)[3] != nrow(B)")
  }

  result_array <- array(0, dim = c(batch_size, seq_len, d_out))

  for (i in 1:batch_size) {
    # A[i, , ] will correctly return a (seq_len, d_in) matrix if seq_len > 1.
    # If seq_len == 1, it might collapse to a vector, which would be an issue.
    # However, for typical transformer sequence lengths, it's usually > 1.
    result_array[i, , ] <- A[i, , ] %*% B
  }
  return(result_array)
}

#' Row-Major Head Splitting (Mimics Python's reshape(-1, num_heads, d_k))
split_heads <- function(X, num_heads) {
  dims <- dim(X)
  b <- dims[1]; s <- dims[2]; d_model <- dims[3]
  d_k <- d_model / num_heads

  # Move d_model to front, split it into (d_k, num_heads), then permute back
  X_perm <- aperm(X, c(3, 2, 1))
  dim(X_perm) <- c(d_k, num_heads, s, b)
  return(aperm(X_perm, c(4, 3, 2, 1))) # Returns: (batch, seq, num_heads, d_k)
}

#' Row-Major Head Merging (Mimics Python's concatenated reshape)
merge_heads <- function(X) {
  dims <- dim(X)
  b <- dims[1]; s <- dims[2]; h <- dims[3]; k <- dims[4]

  # Move (heads, d_k) to front, merge into d_model, permute back
  X_perm <- aperm(X, c(4, 3, 2, 1))
  dim(X_perm) <- c(h * k, s, b)
  return(aperm(X_perm, c(3, 2, 1))) # Returns: (batch, seq, d_model)
}

#' Batched Matrix Multiplication for 4D Arrays
bmm_4d <- function(A, B) {
  b <- dim(A)[1]; h <- dim(A)[2]; seq1 <- dim(A)[3]; seq2 <- dim(B)[4]
  out <- array(0, dim = c(b, h, seq1, seq2))

  # Explicitly loop the batch and head dimensions to prevent R %*% dimension dropping
  for (i in 1:b) {
    for (j in 1:h) {
      out[i, j, , ] <- A[i, j, , ] %*% B[i, j, , ]
    }
  }
  return(out)
}

#' Softmax Activation (Computed precisely along the last dimension)
softmax_last_dim <- function(x) {
  # x is expected to be (batch, num_heads, seq_len, seq_len)
  margins <- c(1, 2, 3) # Apply over the first three dimensions to target the 4th

  x_max <- apply(x, margins, max)
  x_shifted <- sweep(x, margins, x_max, "-")
  e_x <- exp(x_shifted)

  sum_e_x <- apply(e_x, margins, sum)
  res <- sweep(e_x, margins, sum_e_x, "/")

  return(res)
}

# =====================================================================
# MULTI-HEAD ATTENTION IMPLEMENTATION
# =====================================================================

multi_head_attention <- function(Q, K, V, W_q, W_k, W_v, W_o, num_heads) {

  dims <- dim(Q)
  batch_size <- dims[1]; seq_len <- dims[2]; d_model <- dims[3]
  d_k <- d_model / num_heads

  # 1. Linear projections for Q, K, V
  Q_proj <- matmul_3d_2d(Q, W_q)
  K_proj <- matmul_3d_2d(K, W_k)
  V_proj <- matmul_3d_2d(V, W_v)

  # 2. Reshape to separate heads
  Q_heads <- split_heads(Q_proj, num_heads)
  K_heads <- split_heads(K_proj, num_heads)
  V_heads <- split_heads(V_proj, num_heads)

  # 3. Transpose to get (batch, num_heads, seq_len, d_k)
  Q_trans <- aperm(Q_heads, c(1, 3, 2, 4))
  K_trans <- aperm(K_heads, c(1, 3, 2, 4))
  V_trans <- aperm(V_heads, c(1, 3, 2, 4))

  # Transpose K_trans for the dot product -> K^T: (batch, num_heads, d_k, seq_len)
  K_trans_T <- aperm(K_trans, c(1, 2, 4, 3))

  # 4. Scaled dot-product attention
  # Compute attention scores: Q @ K^T -> (batch, num_heads, seq_len, seq_len)
  scores <- bmm_4d(Q_trans, K_trans_T)

  # Scale scores
  scaled_scores <- scores / sqrt(d_k)

  # Apply softmax
  attention_weights <- softmax_last_dim(scaled_scores)

  # Apply attention to values -> (batch, num_heads, seq_len, d_k)
  head_outputs <- bmm_4d(attention_weights, V_trans)

  # 5. Concatenate heads
  # Transpose back to (batch, seq_len, num_heads, d_k)
  head_outputs_trans <- aperm(head_outputs, c(1, 3, 2, 4))

  # Reshape to (batch, seq_len, d_model)
  concatenated <- merge_heads(head_outputs_trans)

  # 6. Final output projection
  output <- matmul_3d_2d(concatenated, W_o)

  return(output)
}


In [6]:
install.packages("torch")
library(torch)
torch::install_torch()

#' Apply position-wise feed-forward network using functional torch
feed_forward_functional <- function(x, W1, b1, W2, b2) {

  # Step 1: First linear transformation: x @ W1 + b1
  # torch_matmul natively handles batched 3D x 2D multiplication.
  # The '+' operator natively broadcasts the 1D bias across the sequence dimension.
  hidden <- torch_matmul(x, W1) + b1

  # Step 2: Apply ReLU activation
  # nnf_relu handles the tensor-wide max(0, hidden) highly efficiently.
  relu_out <- nnf_relu(hidden)

  # Step 3: Second linear transformation: relu_out @ W2 + b2
  output <- torch_matmul(relu_out, W2) + b2

  return(output)
}


Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

also installing the dependencies ‘coro’, ‘safetensors’




In [7]:
layer_norm <- function(x, gamma, beta, eps = 1e-6) {
  #' Apply layer normalization.
  #'
  #' Args:
  #'   x: Input array of shape (..., d_model)
  #'   gamma: Scale parameter of shape (d_model,)
  #'   beta: Shift parameter of shape (d_model,)
  #'   eps: Small constant for numerical stability
  #'
  #' Returns:
  #'   Normalized array of same shape as x

  ndim <- length(dim(x))
  d_model <- dim(x)[ndim]

  # Margins for applying functions across all dimensions except the last one (feature dimension)
  # e.g., if x is (batch, seq_len, d_model), margins_for_stats will be c(1, 2)
  margins_for_stats <- 1:(ndim - 1)

  # Step 1: Compute mean across the feature dimension (last axis)
  # This results in an array of shape dim(x)[1:(ndim-1)]
  mean_val <- apply(x, margins_for_stats, mean)

  # Calculate x_centered = x - mean_val. `sweep` handles broadcasting correctly.
  x_centered <- sweep(x, margins_for_stats, mean_val, FUN = "-")

  # Step 2: Compute population variance across the feature dimension
  # NumPy's np.var default is population variance. R's mean(vec^2) is population variance
  # when vec is already centered.
  variance <- apply(x_centered^2, margins_for_stats, mean)

  # Step 3: Normalize the input
  # Denominator: sqrt(variance + eps)
  std_dev <- sqrt(variance + eps)

  # x_normalized = x_centered / std_dev. `sweep` handles broadcasting correctly.
  x_normalized <- sweep(x_centered, margins_for_stats, std_dev, FUN = "/")

  # Step 4: Apply scale (gamma) and shift (beta)
  # gamma and beta are vectors of length d_model.
  # Use `sweep` along the last dimension (ndim) to apply them element-wise.
  output <- sweep(x_normalized, ndim, gamma, FUN = "*")
  output <- sweep(output, ndim, beta, FUN = "+")

  return(output)
}

# --- Usage Example ---
# Assume input x of shape (batch_size, seq_len, d_model)
batch_size_ln <- 2
seq_len_ln <- 4
d_model_ln <- 3

# Random input data
x_ex_ln <- array(rnorm(batch_size_ln * seq_len_ln * d_model_ln),
                 dim = c(batch_size_ln, seq_len_ln, d_model_ln))

# Gamma and Beta parameters for d_model (typically initialized to ones and zeros)
gamma_ex_ln <- rep(1, d_model_ln) # Scale parameter
beta_ex_ln <- rep(0, d_model_ln)  # Shift parameter

# Apply layer normalization
normalized_output_ln <- layer_norm(x_ex_ln, gamma_ex_ln, beta_ex_ln)

print(paste("Input dimensions:", paste(dim(x_ex_ln), collapse = "x")))
print(paste("Normalized output dimensions:", paste(dim(normalized_output_ln), collapse = "x")))

# Verify mean and population variance of a sample feature vector
# (e.g., first batch, first sequence element's feature vector)
cat("\nVerification for first feature vector (first batch, first sequence element):\n")
sample_vector <- normalized_output_ln[1, 1, ]

cat("  Mean (should be near 0): ", mean(sample_vector), "\n")
# R's var() is sample variance (N-1 denominator). For population variance (N denominator),
# we can calculate it as the mean of squared differences from the mean.
population_variance <- mean((sample_vector - mean(sample_vector))^2)
cat("  Population Variance (should be near 1): ", population_variance, "\n")


[1] "Input dimensions: 2x4x3"
[1] "Normalized output dimensions: 2x4x3"

Verification for first feature vector (first batch, first sequence element):
  Mean (should be near 0):  -5.551115e-17 
  Population Variance (should be near 1):  0.9999989 


In [8]:
library(torch)

#' Provided: Softmax function (Manually translated to torch ops)
softmax <- function(x, dim = -1) {
  # torch_max returns a list of (values, indices), so we extract [[1]]
  max_vals <- torch_max(x, dim = dim, keepdim = TRUE)[[1]]
  e_x <- torch_exp(x - max_vals)
  return(e_x / torch_sum(e_x, dim = dim, keepdim = TRUE))

  # Note: In production, you would just use nnf_softmax(x, dim = dim)
}

#' Apply layer normalization
layer_norm <- function(x, gamma, beta, eps = 1e-6) {
  # Compute mean across feature dimension
  mean_val <- torch_mean(x, dim = -1, keepdim = TRUE)

  # Compute variance. unbiased = FALSE forces population variance, matching np.var
  variance <- torch_var(x, dim = -1, unbiased = FALSE, keepdim = TRUE)

  # Normalize
  x_normalized <- (x - mean_val) / torch_sqrt(variance + eps)

  # Apply scale and shift (Automatic broadcasting applies here)
  output <- gamma * x_normalized + beta

  return(output)
}

#' Multi-head attention
multi_head_attention <- function(Q, K, V, W_q, W_k, W_v, W_o, num_heads) {
  dims <- Q$shape
  batch_size <- dims[1]
  seq_len <- dims[2]
  d_model <- dims[3]
  d_k <- d_model / num_heads

  # 1. Linear projections
  # torch_matmul inherently handles 3D tensors natively
  Q_proj <- torch_matmul(Q, W_q)
  K_proj <- torch_matmul(K, W_k)
  V_proj <- torch_matmul(V, W_v)

  # 2. Reshape to separate heads
  # view() is the torch equivalent of reshape()
  Q_heads <- Q_proj$view(c(batch_size, seq_len, num_heads, d_k))
  K_heads <- K_proj$view(c(batch_size, seq_len, num_heads, d_k))
  V_heads <- V_proj$view(c(batch_size, seq_len, num_heads, d_k))

  # 3. Transpose to get (batch, num_heads, seq_len, d_k)
  # R uses 1-based indexing. We swap seq_len (dim 2) and num_heads (dim 3)
  Q_trans <- Q_heads$permute(c(1, 3, 2, 4))
  K_trans <- K_heads$permute(c(1, 3, 2, 4))
  V_trans <- V_heads$permute(c(1, 3, 2, 4))

  # 4. Scaled dot-product attention
  # Transpose K_trans on its last two dimensions: (batch, num_heads, d_k, seq_len)
  scores <- torch_matmul(Q_trans, K_trans$transpose(3, 4))

  # Scale scores
  scaled_scores <- scores / sqrt(d_k)

  # Apply softmax to get attention weights
  attention_weights <- softmax(scaled_scores, dim = -1)

  # Apply attention to values
  head_outputs <- torch_matmul(attention_weights, V_trans)

  # 5. Concatenate heads
  # Transpose back: (batch, seq_len, num_heads, d_k)
  head_outputs_trans <- head_outputs$permute(c(1, 3, 2, 4))

  # Reshape to (batch, seq_len, d_model)
  # $contiguous() is required before view() because permute() alters memory layout
  concatenated <- head_outputs_trans$contiguous()$view(c(batch_size, seq_len, d_model))

  # 6. Output projection
  output <- torch_matmul(concatenated, W_o)

  return(output)
}

#' Position-wise feed-forward network
feed_forward <- function(x, W1, b1, W2, b2) {
  # First linear layer + bias broadcast
  hidden <- torch_matmul(x, W1) + b1

  # ReLU activation
  relu_out <- nnf_relu(hidden)

  # Second linear layer + bias broadcast
  output <- torch_matmul(relu_out, W2) + b2

  return(output)
}

#' Complete encoder block: MHA + FFN with residuals and layer norms
encoder_block <- function(x, W_q, W_k, W_v, W_o, W1, b1, W2, b2,
                          gamma1, beta1, gamma2, beta2, num_heads) {

  # Part 1: Multi-Head Attention + Residual + Layer Norm
  attn_output <- multi_head_attention(x, x, x, W_q, W_k, W_v, W_o, num_heads)
  x_attn_residual <- x + attn_output
  x_norm1 <- layer_norm(x_attn_residual, gamma1, beta1)

  # Part 2: Feed-Forward Network + Residual + Layer Norm
  ff_output <- feed_forward(x_norm1, W1, b1, W2, b2)
  x_ff_residual <- x_norm1 + ff_output
  output <- layer_norm(x_ff_residual, gamma2, beta2)

  return(output)
}